In [7]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [8]:
def main():
    # 1. Setup our Base Directories
    in_base_dir = os.path.join("result", "attention_timeseries")
    out_base_dir = os.path.join("result", "process", "attention_timeseries")
    
    print(f"[INFO] Scanning {in_base_dir} for all CSV files...")
    
    search_pattern = os.path.join(in_base_dir, "**", "*.csv")
    all_csvs = glob.glob(search_pattern, recursive=True)
    
    if not all_csvs:
        print(f"[ERROR] No CSVs found in {in_base_dir}")
        return
        
    print(f"[INFO] Found {len(all_csvs)} CSV files. Extracting and binning data...")
    
    # 2. Grouping Dictionary for AGGREGATION
    # Key: (model_name, subject, layer) -> Notice we dropped 'instance' and 'prob_id' 
    # because we want to smash all instances and problems together!
    # Value: list of binned dataframes
    master_data_groups = {}
    
    num_bins = 30
    bins = np.linspace(0, 1, num_bins + 1)
    
    for csv_path in tqdm(all_csvs, desc="Reading & Binning CSVs"):
        rel_path = os.path.relpath(csv_path, in_base_dir)
        parts = rel_path.split(os.sep)
        
        if len(parts) < 4:
            continue
            
        model_name = parts[0]
        subject = parts[1]
        filename = parts[-1]
        
        if "_layer" not in filename:
            continue
            
        prefix, layer_part = filename.split("_layer")
        layer = layer_part.replace(".csv", "")
        
        last_underscore_idx = prefix.rfind('_')
        if last_underscore_idx != -1:
            prob_id = prefix[:last_underscore_idx]
            variant = prefix[last_underscore_idx + 1:]
            
            df = pd.read_csv(csv_path)
            if len(df) <= 1:
                continue
                
            # Normalize progress
            min_tok = df['token_number_generated'].min()
            max_tok = df['token_number_generated'].max()
            if max_tok == min_tok:
                continue 
                
            df['normalized_progress'] = (df['token_number_generated'] - min_tok) / (max_tok - min_tok)
            
            # Bin the data
            df['bin'] = pd.cut(df['normalized_progress'], bins=bins, include_lowest=True)
            binned_df = df.groupby('bin', observed=False).mean(numeric_only=True).reset_index()
            binned_df['plot_x'] = binned_df['bin'].apply(lambda x: x.mid)
            binned_df = binned_df.dropna(subset=['IAR_normalized'])
            
            if binned_df.empty:
                continue
                
            # Tag the data so we can group it later
            binned_df['variant'] = variant
            binned_df['prob_id'] = prob_id
            
            key = (model_name, subject, layer)
            if key not in master_data_groups:
                master_data_groups[key] = []
            
            # Keep only what we need to save memory
            master_data_groups[key].append(binned_df[['variant', 'prob_id', 'plot_x', 'IAR_normalized']])
    
    print(f"\n[INFO] Data extracted. Found {len(master_data_groups)} unique (Model, Subject, Layer) combinations to aggregate.")
    
    # 3. Calculate Statistics & Plot
    for key, dfs in tqdm(master_data_groups.items(), desc="Plotting Aggregate Graphs"):
        model_name, subject, layer = key
        
        # Combine all problems for this specific subject/model/layer into one giant dataframe
        master_df = pd.concat(dfs, ignore_index=True)
        num_unique_problems = len(master_df['prob_id'].unique())
        
        # Calculate Mean and Standard Error (sem) across all problems for each variant at each X-bin
        agg_df = master_df.groupby(['variant', 'plot_x'])['IAR_normalized'].agg(['mean', 'sem']).reset_index()
        
        # 95% Confidence Interval = Mean +/- (1.96 * Standard Error)
        agg_df['ci_lower'] = (agg_df['mean'] - (1.96 * agg_df['sem'])).clip(lower=0.0)
        agg_df['ci_upper'] = (agg_df['mean'] + (1.96 * agg_df['sem'])).clip(upper=1.0)
        
        # Prepare output directory
        out_dir = os.path.join(out_base_dir, model_name, subject)
        os.makedirs(out_dir, exist_ok=True)
        
        plt.figure(figsize=(14, 7))
        variants = agg_df['variant'].unique()
        
        for variant in sorted(variants):
            v_data = agg_df[agg_df['variant'] == variant]
            
            # Plot the thick average trendline
            line = plt.plot(
                v_data['plot_x'], 
                v_data['mean'], 
                label=variant.capitalize(),
                linewidth=2.5,
                marker='o',
                markersize=4
            )
            
            # Add the shaded 95% Confidence Interval halo
            plt.fill_between(
                v_data['plot_x'], 
                v_data['ci_lower'], 
                v_data['ci_upper'], 
                alpha=0.15, 
                color=line[0].get_color()
            )

        # Formatting
        plt.title(f"Aggregate Image Attention Ratio (n={num_unique_problems} problems)\nModel: {model_name} | Subject: {subject.capitalize()} | Layer: {layer}", fontsize=14, pad=15)
        plt.xlabel("Generation Progress (0 = Start, 1 = End)", fontsize=12)
        plt.ylabel("Mean-Normalized IAR (with 95% CI)", fontsize=12)
        
        # Lock Y-axis slightly outside 0 and 1 so the shading isn't cut off abruptly
        plt.ylim(-0.05, 1.05)
        plt.grid(True, linestyle='--', alpha=0.6)
        
        plt.legend(title="Language Variant", bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=11)
        plt.tight_layout()
        
        # Save
        out_filename = f"AGGREGATE_TREND_{subject}_layer{layer}.png"
        out_path = os.path.join(out_dir, out_filename)
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()

    print("\n[SUCCESS] Master aggregate graphs generated successfully!")

if __name__ == "__main__":
    main()

[INFO] Scanning result\attention_timeseries for all CSV files...
[INFO] Found 1379 CSV files. Extracting and binning data...


Reading & Binning CSVs: 100%|██████████| 1379/1379 [02:55<00:00,  7.84it/s]



[INFO] Data extracted. Found 1 unique (Model, Subject, Layer) combinations to aggregate.


Plotting Aggregate Graphs:   0%|          | 0/1 [00:00<?, ?it/s]C:\Users\ACER\AppData\Local\Temp\ipykernel_12740\1442605125.py:91: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_df = master_df.groupby(['variant', 'plot_x'])['IAR_normalized'].agg(['mean', 'sem']).reset_index()
Plotting Aggregate Graphs: 100%|██████████| 1/1 [00:02<00:00,  2.25s/it]


[SUCCESS] Master aggregate graphs generated successfully!
